# AdSem Feature Extraction — Wikipedia Background (WSKF-style)

This notebook computes the four **AdSem** features — **Richness**, **Clarity**, **Noise**, and **Topic count** , applied with **Wikipedia English** as the background corpus.


**Output:** `adsem_features.csv` with columns
`BWiki05Rich_S, BWiki05Clar_S, BWiki05Nois_S, BWiki05Topc_S, BWiki10Rich_S, ...`
plus metadata (`original_index`, `label`, `n_tokens`, `short_text_flag`).


## 1 · Install

We pin versions so the paper's results reproduce. Update only if you re-run end-to-end.

In [1]:
# One-time fix for the scipy.linalg.triu ImportError on Colab.
# After this cell finishes, the runtime auto-restarts — that is required because
# gensim/scipy were already partially imported in the broken state.
!pip install -q --upgrade 'gensim>=4.4.0' 'scipy>=1.13'


In [2]:
# Pinned versions for reproducibility.
# If you change any version, re-run the entire notebook.
!pip install -q \
    'datasets==2.19.0' \
    'nltk==3.8.1' \
    'openpyxl==3.1.2' \
    'tqdm==4.66.4' \
    'matplotlib>=3.7'

## 2 · Imports & environment snapshot

Printed versions go into the snapshot JSON we save at the end.

In [3]:
import os, re, json, pickle, hashlib, logging, random, sys, platform
from pathlib import Path

import numpy as np
import pandas as pd
from tqdm.auto import tqdm

import gensim
from gensim import corpora
from gensim.models import LdaModel
from gensim.models.coherencemodel import CoherenceModel

import nltk
from datasets import load_dataset

nltk.download('stopwords', quiet=True)
STOPWORDS = set(nltk.corpus.stopwords.words('english'))

logging.basicConfig(level=logging.INFO, format='%(asctime)s  %(message)s')
log = logging.getLogger()

ENV = {
    'python':  sys.version.split()[0],
    'platform': platform.platform(),
    'gensim':  gensim.__version__,
    'numpy':   np.__version__,
    'pandas':  pd.__version__,
    'nltk':    nltk.__version__,
}
print('Environment:')
for k, v in ENV.items():
    print(f'  {k:<10} {v}')

Environment:
  python     3.12.13
  platform   Linux-6.6.122+-x86_64-with-glibc2.35
  gensim     4.4.0
  numpy      2.0.2
  pandas     2.2.2
  nltk       3.8.1


## 3 · Configuration

**Choose your Wikipedia source carefully** — this is the most important methodological choice and reviewers will ask:

- `bhasha-wiki-indic` — ~208k English-language Wikipedia articles filtered to **India-related content** by Soket Labs (keyword filter + few-shot classifier). Use this if your target documents are NCERT / Indian-context texts. Topic-matched background → cleaner LDA signal.
- `wikimedia/wikipedia` (config `20231101.en`) — canonical full English Wikipedia (~6.4M articles). Use this if your target documents are general English. More established, easier to defend in review.

Set `WIKI_SOURCE` to one of `'bhasha_indic'` or `'canonical'`.

In [27]:
# ── Data paths ───────────────────────────────────────────────────────────────
DATASET_XLSX = '/content/english_supp_14.05.26_cleant.xlsx'
TEXT_COL     = 'content'
LABEL_COL    = 'grade '

# ── Wikipedia background ─────────────────────────────────────────────────────
WIKI_SOURCE  = 'bhasha_indic'        # 'bhasha_indic' | 'canonical'
WIKI_MAX     = 10_000                # n training docs; 10k = stable + fast
WIKI_MIN_TOK = 50                    # drop stubs from LDA training corpus
WIKI_SAMPLE_POOL = 50_000            # stream this many, randomly sample WIKI_MAX
RANDOM_SEED  = 42

# ── LDA ──────────────────────────────────────────────────────────────────────
TOPIC_COUNTS = [50, 100, 150, 200]   # matches LingFeat (Lee et al. 2021) exactly
LDA_PASSES   = 5
LDA_ITERS    = 50
LDA_CHUNK    = 2000

# ── Output ───────────────────────────────────────────────────────────────────
MODEL_DIR    = '/content/lda_models'
OUTPUT_CSV   = '/content/adsem_features_full(check).csv'
SNAPSHOT_JSON = '/content/adsem_snapshot.json'

os.makedirs(MODEL_DIR, exist_ok=True)
random.seed(RANDOM_SEED)
np.random.seed(RANDOM_SEED)
print('Config ready.')

Config ready.


## 4 · Load Wikipedia background corpus

We stream `WIKI_SAMPLE_POOL` candidates, then **randomly sample** `WIKI_MAX` with `RANDOM_SEED`. This avoids the bias of always taking the first K documents from the HF stream (which follows a deterministic, alphabetical-ish ID order).

In [25]:
def load_wiki(source: str, max_docs: int, pool: int, seed: int):
    if source == 'bhasha_indic':
        ds = load_dataset(
            'soketlabs/bhasha-wiki-indic',
            '20231101.en',
            split='train',
            streaming=True,
        )
        text_key = 'text'
    elif source == 'canonical':
        ds = load_dataset(
            'wikimedia/wikipedia',
            '20231101.en',
            split='train',
            streaming=True,
        )
        text_key = 'text'
    else:
        raise ValueError(f'Unknown WIKI_SOURCE: {source}')

    log.info(f'Streaming up to {pool:,} candidates from {source} ...')
    candidates = []
    for i, row in enumerate(tqdm(ds, total=pool, desc='Streaming wiki')):
        txt = row.get(text_key, '')
        if txt:
            candidates.append(txt)
        if i + 1 >= pool:
            break

    rng = random.Random(seed)
    sampled = rng.sample(candidates, min(max_docs, len(candidates)))
    log.info(f'Streamed {len(candidates):,} candidates; sampled {len(sampled):,} for LDA training.')
    return sampled

wiki_texts = load_wiki(WIKI_SOURCE, WIKI_MAX, WIKI_SAMPLE_POOL, RANDOM_SEED)

Streaming wiki:   0%|          | 0/50000 [00:00<?, ?it/s]

## 5 · Load downstream (graded) dataset

In [28]:
df = pd.read_excel(DATASET_XLSX)
print(f'Dataset: {len(df)} rows | columns: {list(df.columns)}')
assert TEXT_COL  in df.columns, f"Column '{TEXT_COL}' not found"
assert LABEL_COL in df.columns, f"Column '{LABEL_COL}' not found"
print(f'Grade distribution:\n{df[LABEL_COL].value_counts().sort_index()}')

Dataset: 172 rows | columns: ['index', 'grade ', 'content']
Grade distribution:
grade 
2    26
3    19
4    21
5    27
6    27
7    26
8    26
Name: count, dtype: int64


## 6 · Preprocessing

Same tokenizer applied to both the LDA training corpus and the target documents.
- Keep alphabetic only.
- Lowercase.
- Drop tokens shorter than 3 characters and standard NLTK English stopwords.
- For LDA training: filter docs with fewer than `WIKI_MIN_TOK` tokens (stubs hurt topic quality).

In [7]:
def preprocess(text: str):
    text = re.sub(r'[^a-zA-Z\s]', ' ', str(text)).lower()
    return [t for t in text.split() if len(t) >= 3 and t not in STOPWORDS]

def build_bow(texts, tag='', min_tok=0):
    tokenised = [preprocess(t) for t in tqdm(texts, desc=f'Tokenising {tag}')]
    if min_tok > 0:
        before = len(tokenised)
        tokenised = [t for t in tokenised if len(t) >= min_tok]
        log.info(f'{tag}: dropped {before - len(tokenised):,} short docs (<{min_tok} tok)')
    d = corpora.Dictionary(tokenised)
    d.filter_extremes(no_below=5, no_above=0.5)
    bow = [d.doc2bow(t) for t in tokenised]
    log.info(f'{tag}: vocab={len(d):,}  docs={len(bow):,}')
    return d, bow, tokenised

print('Preprocessing utilities defined.')

Preprocessing utilities defined.


## 7 · Train LDA models and report c_v coherence

We report **c_v topic coherence** for each K. This is the standard quantitative justification for the chosen topic counts — include this table in your paper's appendix.

In [8]:
def _code(n: int) -> str:
    """50→'05', 100→'10', 150→'15', 200→'20' — matches LingFeat naming."""
    return f'{n // 10:02d}'

def train_lda(bow, dictionary, n_topics, tag):
    path = f'{MODEL_DIR}/{tag}_lda_{n_topics}.model'
    if os.path.exists(path):
        log.info(f'Loading cached: {path}')
        return LdaModel.load(path)
    log.info(f'Training LDA  tag={tag}  K={n_topics} ...')
    m = LdaModel(
        corpus=bow, id2word=dictionary, num_topics=n_topics,
        alpha='symmetric', eta='symmetric',
        passes=LDA_PASSES, iterations=LDA_ITERS,
        random_state=RANDOM_SEED, update_every=1, chunksize=LDA_CHUNK,
    )
    m.save(path)
    log.info(f'Saved: {path}')
    return m

def train_all(texts, tag, min_tok=0):
    d, bow, tokenised = build_bow(texts, tag, min_tok=min_tok)
    pickle.dump(d, open(f'{MODEL_DIR}/{tag}_dict.pkl', 'wb'))
    models, coherences = {}, {}
    for n in TOPIC_COUNTS:
        m = train_lda(bow, d, n, tag)
        models[n] = m
        cm = CoherenceModel(model=m, texts=tokenised, dictionary=d, coherence='c_v')
        coherences[n] = float(cm.get_coherence())
        log.info(f'  K={n:3d}  c_v = {coherences[n]:.4f}')
    return models, d, coherences

log.info(f'=== Training LDA on Wikipedia background ({WIKI_SOURCE}) ===')
models_bwiki, dict_bwiki, coherences_bwiki = train_all(
    wiki_texts, 'BWiki', min_tok=WIKI_MIN_TOK,
)
print('\n=== Topic coherence (c_v) — higher is better ===')
for k, v in coherences_bwiki.items():
    print(f'  K={k:3d}  c_v = {v:.4f}')

Tokenising BWiki:   0%|          | 0/10000 [00:00<?, ?it/s]


=== Topic coherence (c_v) — higher is better ===
  K= 50  c_v = 0.5904
  K=100  c_v = 0.5486
  K=150  c_v = 0.5328
  K=200  c_v = 0.5096


## 8 · AdSem feature functions

Verbatim translation of `OSKF.py`. The three subtle points that any re-implementation must get right:

1. **Descending sort.** `p_0` is the *maximum* probability topic. Both *richness* and *clarity* depend on this ordering.
2. **Gensim default threshold.** Do **not** pass `minimum_probability=0.0`. We want the model's natural "active topic" count, which is what `Topc_S` measures.
3. **Noise uses uniform mean.** μ = 1/n (the uniform baseline), not `np.mean(p)`. With Gensim's default threshold, the returned probabilities don't sum to 1, so `np.mean` would give the wrong reference and the kurtosis interpretation would change.

In [10]:
def get_probability_list(model, bow_doc):
    """Return Gensim's default topic distribution, sorted DESCENDING."""
    raw = model.get_document_topics(bow_doc)   # default threshold applies
    return sorted([p for _, p in raw], reverse=True)


def semantic_richness(p):
    """Σ (idx+1) · p_i with p descending. Captures topical breadth."""
    return sum((i + 1) * v for i, v in enumerate(p))


def semantic_clarity(p):
    """(1/n) · Σ_{i=1..n-1} (p_0 − p_i). Captures dominance of top topic."""
    n = len(p)
    if n == 0: return 0.0
    if n == 1: return p[0]
    total = sum(p[0] - v for v in p[1:])
    return total / n


def semantic_noise(p):
    """n · Σ(p-1/n)^4 / (Σ(p-1/n)^2)^2. Kurtosis vs uniform baseline."""
    n = len(p)
    if n == 0: return 0.0
    mean = 1.0 / n
    s2 = sum((v - mean) ** 2 for v in p)
    s4 = sum((v - mean) ** 4 for v in p)
    denom = s2 ** 2
    if denom == 0.0: return 0.0
    return n * (s4 / denom)

## 9 · Unit tests against hand-calculated values

These tests prove the implementation matches the formulas — keep them green before submitting.

In [11]:
def _unit_tests():
    # richness: [0.6,0.3,0.1] desc → 1*0.6 + 2*0.3 + 3*0.1 = 1.5
    assert abs(semantic_richness([0.6, 0.3, 0.1]) - 1.5) < 1e-9, 'richness failed'

    # clarity: ((0.6-0.3)+(0.6-0.1))/3 = 0.8/3
    assert abs(semantic_clarity([0.6, 0.3, 0.1]) - 0.8/3) < 1e-9, 'clarity failed'
    assert semantic_clarity([1.0]) == 1.0, 'clarity single-element failed'
    assert semantic_clarity([]) == 0.0, 'clarity empty failed'

    # noise: against uniform mean 1/n
    probs = [0.6, 0.3, 0.1]
    mean  = 1/3
    s2 = sum((p - mean)**2 for p in probs)
    s4 = sum((p - mean)**4 for p in probs)
    expected = 3 * s4 / s2**2
    assert abs(semantic_noise(probs) - expected) < 1e-9, 'noise failed'
    assert semantic_noise([]) == 0.0, 'noise empty failed'

    print('✓ All unit tests passed.')

_unit_tests()

✓ All unit tests passed.


## 10 · Feature extraction per document

In [12]:
ALL_MODELS = {
    'BWiki': (models_bwiki, dict_bwiki),
    # Add more background corpora here as (models_dict, dictionary) pairs
    # e.g. 'NCERT': (models_ncert, dict_ncert),
}

def extract_features(text):
    tokens = preprocess(text)
    feats = {}
    for corpus_tag, (models, dictionary) in ALL_MODELS.items():
        bow = dictionary.doc2bow(tokens)
        for n_topics, model in models.items():
            prob_list = get_probability_list(model, bow)
            prefix = f'{corpus_tag}{_code(n_topics)}'
            feats[f'{prefix}Rich_S'] = semantic_richness(prob_list)
            feats[f'{prefix}Clar_S'] = semantic_clarity(prob_list)
            feats[f'{prefix}Nois_S'] = semantic_noise(prob_list)
            feats[f'{prefix}Topc_S'] = len(prob_list)
    return feats

print('Feature extractor ready. Models:', list(ALL_MODELS.keys()),
      '× K =', TOPIC_COUNTS)

Feature extractor ready. Models: ['BWiki'] × K = [50, 100, 150, 200]


## 11 · Extract features for every row and save

In [29]:
rows = []
for idx, row in tqdm(df.iterrows(), total=len(df), desc='Extracting AdSem'):
    feats = extract_features(str(row[TEXT_COL]))
    feats['original_index']  = row['index']
    feats['label']           = row[LABEL_COL]
    n_tok = len(preprocess(str(row[TEXT_COL])))
    feats['n_tokens']        = n_tok
    feats['short_text_flag'] = int(n_tok < 50)
    rows.append(feats)

feat_df = pd.DataFrame(rows)
meta = ['original_index', 'label', 'n_tokens', 'short_text_flag']
feat_df = feat_df[meta + [c for c in feat_df.columns if c not in meta]]
feat_df.to_csv(OUTPUT_CSV, index=False)

print(f'Saved  → {OUTPUT_CSV}')
print(f'Shape  : {feat_df.shape}')

n_short = feat_df['short_text_flag'].sum()
if n_short:
    print(f'⚠  {n_short} rows have <50 tokens — LDA features are unreliable here.')
feat_df.head()

Extracting AdSem:   0%|          | 0/172 [00:00<?, ?it/s]

Saved  → /content/adsem_features_full(check).csv
Shape  : (172, 20)
⚠  5 rows have <50 tokens — LDA features are unreliable here.


,original_index,label,n_tokens,short_text_flag,BWiki05Rich_S,BWiki05Clar_S,BWiki05Nois_S,BWiki05Topc_S,BWiki10Rich_S,BWiki10Clar_S,BWiki10Nois_S,BWiki10Topc_S,BWiki15Rich_S,BWiki15Clar_S,BWiki15Nois_S,BWiki15Topc_S,BWiki20Rich_S,BWiki20Clar_S,BWiki20Nois_S,BWiki20Topc_S
0,1,2,134,0,2.520942,0.230889,3.351323,6,2.307075,0.256389,2.313867,8,3.289445,0.181822,2.635519,10,2.693218,0.345868,6.235917,10
1,2,2,81,0,2.365866,0.242186,2.257184,8,2.965841,0.204199,3.131465,9,4.098769,0.259310,7.953566,17,4.886749,0.094496,1.828839,16
2,3,2,222,0,3.323560,0.189323,3.327697,10,4.634767,0.118007,2.601281,14,3.907680,0.190932,4.393152,14,4.686950,0.276926,12.289189,19
3,4,2,35,1,1.836001,0.277353,1.917540,5,3.601443,0.194772,4.032968,11,3.360673,0.193674,3.152877,11,5.736112,0.067523,2.212357,15
4,5,2,25,1,2.547040,0.102016,2.179284,5,2.435940,0.148949,1.835462,6,2.734659,0.100115,1.644364,6,1.801722,0.236374,1.858272,4
